<a href="https://colab.research.google.com/github/dodi-ctrl/PhishingDetector/blob/main/URL_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# URL Agent — Multi-Corpus Training

Trains the Random Forest URL agent on URLs embedded in **the same three email corpora** as the Metadata Agent and the augmented DistilBERT text agent. This keeps the training data consistent across all three agents.

| Source | Label | Provides |
|---|---|---|
| `phishing_pot` (GitHub) | phishing (1) | URLs inside modern 2022–2024 phishing emails |
| Nazario `phishing3.mbox` (≥2022) | phishing (1) | URLs from honeypot phishing |
| Enron ham via HuggingFace | legitimate (0) | URLs inside legitimate corporate mail |

In [ ]:
# Install dependencies
!pip install -q datasets scikit-learn pandas numpy matplotlib seaborn joblib
print("Installation OK")

In [ ]:
# Clone the repo
import os

if not os.path.exists('PhishingDetector'):
    !git clone https://github.com/dodi-ctrl/PhishingDetector.git

os.chdir('PhishingDetector')
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))

## Download phishing corpora

- **phishing_pot**: cloned from GitHub — thousands of recent .eml files in `phishing_pot/email/`.
- **Nazario phishing3.mbox**: ~3,000 phishing messages.
- **Enron ham** is fetched on demand from HuggingFace inside `build_eml_corpus`.

In [ ]:
import os

if not os.path.exists('phishing_pot'):
    !git clone --depth 1 https://github.com/rf-peixoto/phishing_pot.git
if not os.path.exists('phishing3.mbox'):
    !wget -q https://monkey.org/~jose/phishing/phishing3.mbox

for f in ('phishing_pot', 'phishing3.mbox'):
    print(f'  {f}: {"OK" if os.path.exists(f) else "MISSING"}')

In [ ]:
# Build EML corpus (for email-embedded URLs)
from dataset_handling import build_eml_corpus, build_url_corpus, extract_url_features_from_url_corpus
from feature_extraction import FeatureExtractor

PHISHING_DIRS = ['phishing_pot/email', 'phishing_pot/emails']
PHISHING_MBOX = ['phishing3.mbox']
ENRON_MAX     = 5000

eml_df = build_eml_corpus(
    phishing_dirs=PHISHING_DIRS,
    phishing_mbox_paths=PHISHING_MBOX,
    enron_max=ENRON_MAX,
    nazario_after_year=2022,
)

In [ ]:
# Build the URL corpus from .eml bodies
url_df = build_url_corpus(eml_df=eml_df if len(eml_df) > 0 else None)

In [ ]:
# Extract URL features
extractor = FeatureExtractor()
features_df, labels = extract_url_features_from_url_corpus(url_df, extractor)

print('\nFeature matrix shape:', features_df.shape)
print('URL feature columns:', features_df.columns.tolist())

In [ ]:
# Train / test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_df, labels,
    test_size=0.3,
    random_state=42,
    stratify=labels,
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"  - Legitimate:   {(y_test == 0).sum()}")
print(f"  - Phishing:     {(y_test == 1).sum()}")

In [ ]:
# Train the URL Agent
from url_agent import URLAgent

agent = URLAgent(n_estimators=100, max_depth=20, random_state=42)
agent.train(X_train, y_train, validate=True, tune_hyperparameters=False)

In [ ]:
# Evaluate — accuracy, precision, recall, F1, ROC-AUC, FPR/FNR/TNR,
# confusion matrix, classification report, and plots
results = agent.evaluate(X_test, y_test, plot_results=True)

In [ ]:
# Save model and results
agent.save_model('url_agent.pkl')
agent.export_results(results, 'url_agent_results.json')

print('\n' + '=' * 70)
print('Training complete!')
print(f"  Accuracy:  {results['accuracy']:.4f}")
print(f"  F1-Score:  {results['f1_score']:.4f}")
print(f"  ROC-AUC:   {results['roc_auc']:.4f}")
print('=' * 70)